In [128]:
#Q1. Data Ingestion Pipeline
#1.Read all datasets.

import pandas as pd
df_customers = pd.read_csv('customers.csv')
df_orders = pd.read_csv('orders.csv')
df_products = pd.read_csv('products.csv')
df_order_items = pd.read_csv('order_items.csv')
df_events = pd.read_csv('events.csv')
df_sessions = pd.read_csv('sessions.csv')


In [129]:
#2.Validate schema consistency.

expected_customers = {
    "customer_id": "int64",
    "name": "object",
    "email": "object",
    "country": "object",
    "age": "int64",
    "signup_date": "object",
    "marketing_opt_in": "bool"
}

expected_orders = {
    "order_id": "int64",
    "customer_id": "int64",
    "order_time": "object", 
    "payment_method": "object",
    "discount_pct": "int64", 
    "subtotal_usd": "float64",
    "total_usd": "float64",
    "country": "object", 
    "device": "object",
    "source": "object"
}

def schema_validation (df, expected_schema):
    print("Starting schema validation")
    for col, dtype in expected_schema.items():
        if col not in df.columns:
            print(f"Missing columns: {col}")
        elif df[col].dtype != dtype:
            print (f" Column: {col} data type is {df[col].dtype} but it should be {dtype}")
    print("Schema validation completed")

schema_validation(df_customers, expected_customers)
schema_validation(df_orders, expected_orders)

Starting schema validation
Schema validation completed
Starting schema validation
Schema validation completed


In [130]:
# 3. Detect duplicate customers.

def duplicated_customers(df):
    print("Checking duplicated Customer IDs")
    duplicated_ids = df[
        df.duplicated(subset=["customer_id"], keep=False)
    ]
    if not duplicated_ids.empty:
        print("\nDuplicated Customer IDs:")
        print(duplicated_ids)
    else:
        print("\nNo duplicated Customer IDs")
    
    print("\nChecking duplicated Emails")
    
    duplicated_emails = df[
        df.duplicated(subset=["email"], keep=False)
    ]
    if not duplicated_emails.empty:
        print("\nDuplicated Customers by Email:")
        print(duplicated_emails)
    else:
        print("\nNo duplicated Emails")
        
    print("\nDuplicated Customers search completed")

duplicated_customers(df_customers)
    

Checking duplicated Customer IDs

No duplicated Customer IDs

Checking duplicated Emails

No duplicated Emails

Duplicated Customers search completed


In [114]:
# 4. Identify invalid records.

def invalid_records ():

    print(df_orders.isnull().sum())
    print("\nChecking for invalid orders in progress...")
    invalid_orders = df_orders[(df_orders["subtotal_usd"] <= 0) | (df_orders["total_usd"] <= 0)]
    if not invalid_orders.empty:
        print("\nInvalid Orders:")
        print(invalid_orders)
    
    orders_without_customer = df_orders[df_orders["customer_id"].isnull()]
    if not orders_without_customer.empty:
        print("Orders without customers:")
        print(orders_without_customer)    
    print("\nChecking for invalid orders completed")
    
    print(df_customers.isnull().sum())
    print("\nChecking for invalid customers in progress...")
    invalid_customers = df_customers[(df_customers["name"].isnull()) | (df_customers["email"].isnull())]
    if not invalid_customers.empty:
        print("\nInvalid Customers:")
        print(invalid_customers)
    print("\nChecking for invalid customers completed")

invalid_records()
    

    

order_id          0
customer_id       0
order_time        0
payment_method    0
discount_pct      0
subtotal_usd      0
total_usd         0
country           0
device            0
source            0
dtype: int64

Checking for invalid orders in progress...

Checking for invalid orders completed
customer_id         0
name                0
email               0
country             0
age                 0
signup_date         0
marketing_opt_in    0
dtype: int64

Checking for invalid customers in progress...

Checking for invalid customers completed


In [131]:
#Q2. Data Cleaning & Transformation
def cleaning_transformation():
    print("Cleaning and transformation in progres...")
    
    # Clean country values
    df_customers["country"] = (df_customers["country"].str.strip().str.upper())

    # Convert datetime columns
    df_orders["order_time"] = pd.to_datetime(df_orders["order_time"],errors="coerce")
    df_events["timestamp"] = pd.to_datetime(df_events["timestamp"],errors="coerce")

    # Remove invalid rows
    df_orders.dropna(subset=["order_id", "total_usd"],inplace=True)
    df_customers.dropna(subset=["name"],inplace=True)
    df_products.dropna(subset=["name"],inplace=True)
    df_events.dropna(subset=["event_type"],inplace=True)
    df_sessions.dropna(subset=["session_id"],inplace=True)

    # Fill missing values
    df_orders["customer_id"] = df_orders["customer_id"].fillna("Unknown")
    df_customers["email"] = df_customers["email"].fillna("Unknown")
    df_products["category"] = df_products["category"].fillna("Unknown")
    df_events["product_id"] = df_events["product_id"].fillna(0)

    # Remove duplicates
    df_orders.drop_duplicates(inplace=True)
    df_customers.drop_duplicates(inplace=True)
    df_products.drop_duplicates(inplace=True)
    df_events.drop_duplicates(inplace=True)
    df_order_items.drop_duplicates(inplace=True)
    df_sessions.drop_duplicates(inplace=True)

    # Export cleaned files
    df_customers.to_csv("clean_customers.csv",index=False)
    df_orders.to_csv("clean_orders.csv",index=False)
    df_products.to_csv("clean_products.csv",index=False)
    df_events.to_csv("clean_events.csv",index=False)
    df_order_items.to_csv("clean_order_items.csv",index=False)
    df_sessions.to_csv("clean_sessions.csv",index=False)

    df_customers.to_parquet("clean_customers.parquet",index=False)
    df_orders.to_parquet("clean_orders.parquet",index=False)
    df_products.to_parquet("clean_products.parquet",index=False)
    df_events.to_parquet("clean_events.parquet",index=False)
    df_order_items.to_parquet('clean_order_items.parquet',index=False)
    df_sessions.to_parquet("clean_sessions.parquet",index=False)

    print("Cleaning and transformation completed.")


cleaning_transformation()

Cleaning and transformation in progres...
Cleaning and transformation completed.


In [132]:
#Q3. Business KPI Generation
def kpi_generation():

    #Read clean files
    df_orders = pd.read_parquet('clean_orders.parquet')
    df_products = pd.read_parquet('clean_products.parquet')
    df_order_items = pd.read_parquet('clean_order_items.parquet')
    
    
    #Daily sales KPI 
    df_orders["date"] =df_orders["order_time"].dt.date
    daily_sales_kpi = df_orders.groupby("date").agg(
        daily_sales = ("total_usd", "sum")).reset_index().sort_values("date")
    
    #Customer lifetime value
    customer_lifetime_value = df_orders.groupby("customer_id").agg(
        sales_costumer = ("total_usd", "sum")).reset_index()

    #Category-wise revenue
    orders_products = df_orders.merge(df_order_items,
        on="order_id",
        how="left").merge(df_products,
            on="product_id",
            how="left")
    category_wise_revenue = orders_products.groupby("category").agg(
        category_revenue = ("total_usd", "sum")).reset_index()
    
    #Repeat customer percentage
    customer_orders = df_orders.groupby("customer_id").agg(
        total_orders = ("order_id", "count")).reset_index()
    repeat_customers = customer_orders[customer_orders["total_orders"] > 1]
    repeat_customer_percentage = (len(repeat_customers)/len(customer_orders)) * 100                 
    print(f"Repeat Customer Percentage: {repeat_customer_percentage}%")

    # Export aggregated files      
    daily_sales_kpi.to_parquet("daily_sales_kpi.parquet",index=False)
    customer_lifetime_value.to_parquet("customer_lifetime_value.parquet",index=False)
    category_wise_revenue.to_parquet("category_wise_revenue.parquet",index=False)      
      
kpi_generation()

Repeat Customer Percentage: 61.74698795180723%


In [139]:
#Q4. Clickstream Analytics
def clickstream_analytics ():
    #Find most visited pages.
    df_events = pd.read_parquet('clean_events.parquet')
    
    page_views = df_events[df_events["event_type"] == "page_view"]
    product_visited = page_views.groupby("product_id").agg(
        views = ("event_id", "count")).reset_index().sort_values("views", ascending=False)
    most_visited = product_visited.head(10)
    most_visited.to_parquet("most_visited_pages.parquet",index=False)

    #Calculate session counts.
    total_sessions = df_events["session_id"].nunique()
    print("Total Sessions:", total_sessions)
    
    #Find bounce rate.
    session_counts = df_events.groupby("session_id").agg(
    event_count = ("event_id", "count")).reset_index()
    bounced = session_counts[session_counts["event_count"] == 1]
    bounce_rate = (len(bounced) / total_sessions) * 100
    print(f"Bounce Rate is: {bounce_rate}%")
    
    #Find mobile vs desktop traffic percentage.
    df_sessions = pd.read_parquet('clean_sessions.parquet')
    events_sessions = session_counts.merge(df_sessions,
        on="session_id",
        how="left")
    total_traffic = events_sessions["event_count"].sum()
    device_traffic = events_sessions.groupby('device').agg(
        device_sessions =("event_count", "sum")).reset_index()
    device_traffic["traffic %"] = 100 * device_traffic["device_sessions"] / total_traffic
    device_traffic.to_parquet("device_traffic.parquet",index=False)
    
clickstream_analytics ()


Total Sessions: 120000
Bounce Rate is: 9.184166666666666%


In [143]:
#Q5. Export Optimization

#Compare storage sizes.
import os
csv_size = os.path.getsize("clean_events.csv") / 1024  # KB
parquet_size = os.path.getsize("clean_events.parquet") / 1024  # KB

print(f"CSV Size: {csv_size:.2f} KB")
print(f"Parquet Size: {parquet_size:.2f} KB")

#Compare read performance.
import time

start = time.time()
df_csv = pd.read_csv("clean_events.csv")
csv_time = time.time() - start

start = time.time()
df_parquet = pd.read_parquet("clean_events.parquet")
parquet_time = time.time() - start

print(f"CSV Load Time: {csv_time:.2f} sec")
print(f"Parquet Load Time: {parquet_time:.2f} sec")

"""Explain why Parquet is better for analytics:
CSV is row_based storage whlist parquet is colmn_based storage.
Parquet only reads required columns instead of scanning the whole table. This makes queries to be processed much faster.
On the other hand, parquet uses compression improving the storage and resources efficiency. """

CSV Size: 41946.07 KB
Parquet Size: 12953.14 KB
CSV Load Time: 6.74 sec
Parquet Load Time: 1.01 sec


'Explain why Parquet is better for analytics:\nCSV is row_based storage whlist parquet is colmn_based storage.\nParquet only reads required columns instead of scanning the whole table. This makes queries to be processed much faster.\nOn the other hand, parquet uses compression improving the storage and resources efficiency. '